In [1]:
import sys
sys.path.insert(0, "./NeuroLM")

from model.model_vq import VQ
from model.model_neural_transformer import NTConfig

import torch

/home/test/Documents/Projects/EEGSSL/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
vq_path = ".weights/NeuroLm/checkpoints/VQ.pt"
vq = torch.load(vq_path, map_location="cpu", weights_only=False)

In [3]:
encoder_config = NTConfig(**vq["encoder_args"])
decoder_config = NTConfig(**vq["decoder_args"])

model = VQ(
    encoder_config=encoder_config,
    decoder_config=decoder_config
)

{}
Final encoder config NTConfig(block_size=1024, patch_size=200, num_classes=0, in_chans=1, out_chans=16, use_mean_pooling=True, init_scale=0.001, n_layer=12, n_head=12, n_embd=768, dropout=0.0, bias=False)
Final decoder config NTConfig(block_size=1024, patch_size=200, num_classes=0, in_chans=128, out_chans=16, use_mean_pooling=True, init_scale=0.001, n_layer=4, n_head=12, n_embd=768, dropout=0.0, bias=False)


In [4]:
state_dict = vq["model"]

clean_state_dict = {}

for k, v in state_dict.items():
    if k.startswith("_orig_mod.VQ."):
        new_k = k[len("_orig_mod.VQ."):]
        clean_state_dict[new_k] = v
        
model.load_state_dict(clean_state_dict)

<All keys matched successfully>

In [5]:
model

VQ(
  (encoder): NeuralTransformer(
    (patch_embed): TemporalConv(
      (conv1): Conv2d(1, 16, kernel_size=(1, 15), stride=(1, 8), padding=(0, 7))
      (gelu1): GELU(approximate='none')
      (norm1): GroupNorm(4, 16, eps=1e-05, affine=True, bias=True)
      (conv2): Conv2d(16, 16, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
      (gelu2): GELU(approximate='none')
      (norm2): GroupNorm(4, 16, eps=1e-05, affine=True, bias=True)
      (conv3): Conv2d(16, 16, kernel_size=(1, 3), stride=(1, 1), padding=(0, 1))
      (norm3): GroupNorm(4, 16, eps=1e-05, affine=True, bias=True)
      (gelu3): GELU(approximate='none')
      (l): Sequential(
        (0): Linear(in_features=400, out_features=768, bias=True)
        (1): GELU(approximate='none')
      )
    )
    (pos_embed): Embedding(256, 768)
    (time_embed): Embedding(64, 768)
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (ln_1): LayerNorm()
        (attn): CausalSelfAttention(
          (c_attn): Linear(in_featu

In [ ]:
# out[1].shape

In [ ]:
x = torch.randn(4, 1000, 200)

y_raw = torch.randn(4, 1000, 200)

input_mask = torch.zeros(4, 1000)

input_chans = torch.zeros(4, 1000, dtype=torch.long)

input_time = torch.zeros(4, 1000, dtype=torch.long)

y_freq = torch.tensor([250.])

with torch.no_grad():
    comp = model(
        y_raw=y_raw,
        y_freq=y_freq,
        x=x,
        input_mask=input_mask,
        input_chans=input_chans,
        input_time=input_time
    )

In [ ]:
comp[1].shape